![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# From Scratch to Pretrained — Stable Diffusion

Last time we built a diffusion model from scratch on anime faces. Same UNet, same noise schedule, same MSE loss. Today we use a real pretrained one — **Stable Diffusion**. The architecture is the same idea, but it works in a compressed latent space (using a VAE you already know) and takes text prompts as input (using CLIP embeddings you already know).

**What we will do:**
1. Load a pretrained Stable Diffusion model and generate images from text
2. Take apart the pipeline — see the VAE, UNet, text encoder, and scheduler
3. Run the denoising loop manually (same as Day 24, just in latent space)
4. Fine-tune it with LoRA to learn a new visual concept

**Model:** `lambdalabs/miniSD-diffusers` — a small Stable Diffusion variant that fits on free Colab T4.

**Runtime:** ~30 minutes total on T4 GPU

In [ ]:
!pip install diffusers transformers accelerate peft datasets -q

In [ ]:
import torch                                         # core tensor library
import torch.nn.functional as F                      # functions (mse_loss, etc.)
import torch.optim as optim                          # optimizers (AdamW)
from torch.utils.data import DataLoader, Dataset     # data loading
from diffusers import StableDiffusionPipeline        # the full text-to-image pipeline
from diffusers import DDPMScheduler                  # noise scheduler
from peft import LoraConfig, get_peft_model          # LoRA (same as HuBERT/CLIP labs)
from datasets import load_dataset                    # HuggingFace datasets
import torchvision.transforms as T                   # image transforms
from PIL import Image                                # image loading
import matplotlib.pyplot as plt                      # plotting
import numpy as np                                   # numerical ops
from tqdm import tqdm                                # progress bars

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==================== GLOBAL CONFIG ====================
BASE_MODEL = "lambdalabs/miniSD-diffusers"   # small SD model (fits on free T4)
SEED = 42                                     # reproducibility

In [ ]:
# ============================================================
# HELPER: Display a grid of PIL images with titles
# ============================================================

def show_images(images, titles=None, ncols=4, figsize=(14, 7)):
    """Display a list of PIL images in a grid."""
    nrows = (len(images) + ncols - 1) // ncols       # ceiling division
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()                   # handle single row
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i])
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=9, wrap=True)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def plot_loss(losses, title="Training Loss"):
    """Plot a loss curve."""
    plt.figure(figsize=(8, 3))
    plt.plot(losses, color='steelblue', alpha=0.7)
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Part 1: Text to Image in One Line

The `diffusers` library wraps the entire diffusion pipeline into one object. You give it a text prompt, it returns an image. Under the hood it does everything we built by hand in Day 24:

1. **Tokenize** the text prompt into token IDs
2. **Encode** the tokens into embedding vectors (using a CLIP text encoder)
3. **Start from random noise** in latent space
4. **Denoise** step by step with the UNet (conditioned on the text embeddings)
5. **Decode** the final latent back to a full image with the VAE

But from the outside, it is one function call.

In [ ]:
# ============================================================
# LOAD THE PRETRAINED STABLE DIFFUSION PIPELINE
# ============================================================

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,                                      # model name on HuggingFace
    torch_dtype=torch.float16,                       # half precision (saves memory)
)
pipe = pipe.to(device)                               # move everything to GPU
pipe.safety_checker = None                           # disable NSFW filter for class
print("Pipeline loaded!")

In [ ]:
# ============================================================
# GENERATE YOUR FIRST IMAGE FROM TEXT
# ============================================================

prompt = "a photo of a cat sitting on a couch"

image = pipe(
    prompt,                                          # the text description
    num_inference_steps=30,                          # how many denoising steps (more = better, slower)
    guidance_scale=7.5,                              # how strongly to follow the text (higher = more literal)
    generator=torch.Generator(device).manual_seed(SEED),  # fixed seed for reproducibility
).images[0]                                          # .images returns a list, take the first

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title(f'"{prompt}"', fontsize=11)
plt.axis('off')
plt.show()
print(f"Image size: {image.size}")

In [ ]:
# ============================================================
# GENERATE A GRID: different prompts, different images
# ============================================================

prompts = [
    "a photo of a golden retriever in a park",
    "a watercolor painting of a mountain landscape",
    "a photo of a red sports car on a highway",
    "an astronaut riding a horse on the moon",
    "a bowl of fresh fruit on a wooden table",
    "a castle on a cliff overlooking the ocean",
    "a robot reading a book in a library",
    "a snowy village at night with warm lights",
]

images = []
for i, p in enumerate(tqdm(prompts, desc="Generating")):
    img = pipe(
        p, num_inference_steps=30, guidance_scale=7.5,
        generator=torch.Generator(device).manual_seed(SEED + i),
    ).images[0]
    images.append(img)

show_images(images, titles=prompts, ncols=4, figsize=(16, 8))

In [ ]:
# ============================================================
# SAME PROMPT, DIFFERENT SEEDS: the stochastic nature
# ============================================================
# Diffusion starts from random noise. Different noise = different image.
# The prompt stays the same, only the seed changes.

prompt = "a photo of a zebra in the African savanna"
seed_images = []
seed_titles = []
for s in range(4):
    img = pipe(
        prompt, num_inference_steps=30, guidance_scale=7.5,
        generator=torch.Generator(device).manual_seed(s),
    ).images[0]
    seed_images.append(img)
    seed_titles.append(f"seed={s}")

show_images(seed_images, titles=seed_titles, ncols=4, figsize=(14, 4))
print(f'All from the same prompt: "{prompt}"')

---
## Part 2: Taking Apart the Pipeline

In Day 24 you built each piece yourself: the UNet, the noise schedule, the training loop. Stable Diffusion has the same pieces, just bigger and pretrained on millions of images. Let's look at each one.

```
StableDiffusionPipeline
├── tokenizer      — converts text to token IDs (like you used in NLP week)
├── text_encoder   — CLIP text encoder: tokens → embedding vectors
├── vae            — same idea as Day 24 Part 1: compress image ↔ latent
├── unet           — same idea as Day 24 Part 3: predict noise from noisy latent + text
└── scheduler      — noise schedule (like your beta / alpha / alpha_bar)
```

In [ ]:
# ============================================================
# HOW BIG IS EACH COMPONENT?
# ============================================================

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print("Stable Diffusion components:")
print(f"  Text encoder (CLIP): {count_params(pipe.text_encoder):>12,} parameters")
print(f"  UNet (denoiser):     {count_params(pipe.unet):>12,} parameters")
print(f"  VAE (encode/decode): {count_params(pipe.vae):>12,} parameters")
print(f"  {'─' * 45}")
total = count_params(pipe.text_encoder) + count_params(pipe.unet) + count_params(pipe.vae)
print(f"  Total:               {total:>12,} parameters")
print(f"\nFor comparison, your Day 24 U-Net had ~12M parameters.")
print(f"The text encoder is CLIP -- the same model you used for image search!")

In [ ]:
# ============================================================
# THE VAE: Compress an image to latent space and back
# ============================================================
# In Day 24, your VAE compressed 64x64 images to a 128-dim vector.
# The SD VAE compresses 256x256 images to a 4x32x32 latent tensor.
# That is an 8x spatial compression -- way more efficient.

vae = pipe.vae                                       # grab the VAE from the pipeline

# Take a generated image and encode it to latent space
test_image = pipe(
    "a photo of a sunflower", num_inference_steps=30,
    generator=torch.Generator(device).manual_seed(0),
).images[0]

# Convert PIL image to tensor: (1, 3, H, W) in [-1, 1]
img_tensor = T.ToTensor()(test_image).unsqueeze(0).to(device).half() * 2 - 1

# Encode: image → latent
with torch.no_grad():
    latent = vae.encode(img_tensor).latent_dist.sample()  # sample from distribution
    latent = latent * vae.config.scaling_factor            # scale to match training

print(f"Image shape:  {img_tensor.shape}  (3 channels, {img_tensor.shape[2]}x{img_tensor.shape[3]})")
print(f"Latent shape: {latent.shape}  (4 channels, {latent.shape[2]}x{latent.shape[3]})")
print(f"Compression:  {img_tensor.numel() / latent.numel():.0f}x fewer numbers")

# Decode: latent → image
with torch.no_grad():
    decoded = vae.decode(latent / vae.config.scaling_factor).sample
    decoded = (decoded / 2 + 0.5).clamp(0, 1)             # back to [0, 1]

# Show original vs reconstruction
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(test_image)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(decoded[0].permute(1, 2, 0).float().cpu().numpy())
axes[1].set_title('VAE Encode → Decode')
axes[1].axis('off')
plt.suptitle('The VAE compresses to latent space and reconstructs with minimal loss', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# THE TEXT ENCODER: Text prompt → embedding vectors
# ============================================================
# This is a CLIP text encoder -- the same model you used in the CLIP lab
# for zero-shot classification. Here it feeds into the UNet as conditioning.

tokenizer = pipe.tokenizer                           # converts text to token IDs
text_encoder = pipe.text_encoder                     # converts token IDs to embeddings

prompt = "a photo of a cat sitting on a couch"

# Step 1: Tokenize -- text string → list of integer token IDs
tokens = tokenizer(
    prompt,
    padding="max_length",                            # pad to fixed length
    max_length=tokenizer.model_max_length,           # 77 tokens max
    truncation=True,
    return_tensors="pt",                             # return PyTorch tensors
)
input_ids = tokens.input_ids.to(device)              # (1, 77)
print(f"Token IDs shape: {input_ids.shape}")
print(f"First 15 tokens: {input_ids[0, :15].tolist()}")

# Step 2: Encode -- token IDs → embedding vectors
with torch.no_grad():
    text_embeddings = text_encoder(input_ids)[0]     # (1, 77, 768)
print(f"Text embeddings shape: {text_embeddings.shape}")
print(f"Each of the 77 token positions gets a 768-dim embedding vector.")
print(f"The UNet receives these embeddings via cross-attention at every denoising step.")

---
## Part 3: The Denoising Loop (Doing It Yourself)

The pipeline hides the loop. Let's run it manually so you can see it is exactly the same as Day 24:

1. **Start from pure noise** in latent space
2. **For each timestep** (from noisy to clean):
   - Feed the noisy latent + text embedding into the UNet
   - UNet predicts the noise
   - Scheduler removes the predicted noise → slightly cleaner latent
3. **Decode** the final clean latent with the VAE

We will save intermediate images to watch the denoising happen.

In [ ]:
# ============================================================
# MANUAL DENOISING LOOP: same idea as Day 24, in latent space
# ============================================================

prompt = "a photo of a giraffe in the African savanna"
num_steps = 30                                       # denoising steps

# --- Step 1: Encode the text prompt ---
tokens = tokenizer(prompt, padding="max_length",
                   max_length=tokenizer.model_max_length,
                   truncation=True, return_tensors="pt")
with torch.no_grad():
    text_emb = text_encoder(tokens.input_ids.to(device))[0]  # (1, 77, 768)

# For classifier-free guidance we also need unconditional embeddings (empty prompt)
uncond_tokens = tokenizer("", padding="max_length",
                          max_length=tokenizer.model_max_length,
                          truncation=True, return_tensors="pt")
with torch.no_grad():
    uncond_emb = text_encoder(uncond_tokens.input_ids.to(device))[0]

# Concatenate: [unconditional, conditional] for classifier-free guidance
text_emb = torch.cat([uncond_emb, text_emb])        # (2, 77, 768)

# --- Step 2: Start from random noise ---
scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")
scheduler.set_timesteps(num_steps)                   # configure step schedule

generator = torch.Generator(device).manual_seed(SEED)
latent_shape = (1, 4, 32, 32)                        # miniSD latent size
latents = torch.randn(latent_shape, generator=generator,
                       device=device, dtype=torch.float16)
latents = latents * scheduler.init_noise_sigma       # scale initial noise

# --- Step 3: Denoise step by step ---
guidance_scale = 7.5
snapshots = []                                        # save intermediate images
snapshot_steps = [0, 5, 10, 15, 20, 25, 29]          # which steps to save

unet = pipe.unet
for i, t in enumerate(tqdm(scheduler.timesteps, desc="Denoising")):
    # Duplicate latent for classifier-free guidance (uncond + cond)
    latent_input = torch.cat([latents] * 2)           # (2, 4, 32, 32)
    latent_input = scheduler.scale_model_input(latent_input, t)

    # UNet predicts noise
    with torch.no_grad():
        noise_pred = unet(latent_input, t, encoder_hidden_states=text_emb).sample

    # Classifier-free guidance: blend unconditional and conditional predictions
    noise_uncond, noise_cond = noise_pred.chunk(2)    # split back
    noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)

    # Scheduler step: remove predicted noise
    latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Save snapshot at selected steps
    if i in snapshot_steps:
        with torch.no_grad():
            snap = vae.decode(latents / vae.config.scaling_factor).sample
            snap = (snap / 2 + 0.5).clamp(0, 1)
            snap_pil = T.ToPILImage()(snap[0].float().cpu())
            snapshots.append((i, snap_pil))

print("Done! Showing denoising progression:")

In [ ]:
# ============================================================
# VISUALIZE THE DENOISING PROCESS
# ============================================================
# Watch the image emerge from noise -- same progression as Day 24!

fig, axes = plt.subplots(1, len(snapshots), figsize=(18, 4))
for ax, (step, img) in zip(axes, snapshots):
    ax.imshow(img)
    ax.set_title(f"Step {step}/{num_steps-1}", fontsize=10)
    ax.axis('off')
plt.suptitle(f'Manual denoising: "{prompt}"', fontsize=12)
plt.tight_layout()
plt.show()

Looks familiar? It is the exact same loop from Day 24:

| Day 24 (from scratch) | Today (pretrained) |
|---|---|
| `noise = unet(x_t, t)` | `noise = unet(latent, t, text_emb)` |
| Manual formula: `x_{t-1} = ...` | `scheduler.step(noise, t, latent)` |
| Pixel space (3, 64, 64) | Latent space (4, 32, 32) |
| No conditioning | Text conditioning via cross-attention |

The math is identical. The only differences are: (1) we work in compressed latent space, and (2) the UNet also receives text embeddings through cross-attention layers.

---
## Part 4: Fine-Tuning with LoRA

The model generates generic images from its pretraining. What if we want it to learn a new visual style or concept? We could retrain all 860M+ UNet parameters, but that is slow and needs tons of data.

Instead we use **LoRA** — the same technique you applied to HuBERT (audio) and CLIP (vision). We freeze everything and add small trainable rank matrices to the UNet's attention layers. Less than 1% of the parameters are trainable.

**Dataset:** We will use a small set of Naruto anime character images with captions so the model learns to generate anime-style characters from text.

**Training loop:** Same as Day 24. Encode image → add noise → predict noise → MSE loss. The only addition is text conditioning.

In [ ]:
# ==================== FINE-TUNING CONFIG ====================
LORA_R = 8              # LoRA rank (same as HuBERT/CLIP labs)
LORA_ALPHA = 16         # LoRA scaling factor
LORA_DROPOUT = 0.1      # dropout for regularization
TRAIN_STEPS = 1500      # training steps (~15 min on T4)
TRAIN_LR = 1e-4         # learning rate
TRAIN_BATCH = 4         # batch size (fits T4 with LoRA)
RESOLUTION = 256        # image resolution for training

In [ ]:
# ============================================================
# LOAD THE NARUTO DATASET FROM HUGGINGFACE
# ============================================================

hf_dataset = load_dataset("lambdalabs/naruto-blip-captions", split="train")
print(f"Dataset size: {len(hf_dataset)} images")
print(f"Columns: {hf_dataset.column_names}")

# Show some samples
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    sample = hf_dataset[i]
    ax.imshow(sample["image"])
    ax.set_title(sample["text"][:40] + "...", fontsize=8)
    ax.axis('off')
plt.suptitle('Naruto Dataset: images + text captions', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# WRAP INTO A PYTORCH DATASET
# ============================================================

class TextImageDataset(Dataset):
    """Wraps a HuggingFace dataset for diffusion training.
    Returns (image_tensor, caption_string) pairs.
    """
    def __init__(self, hf_dataset, resolution=RESOLUTION):
        self.data = hf_dataset
        self.transform = T.Compose([
            T.Resize((resolution, resolution)),      # resize to fixed size
            T.CenterCrop(resolution),                # crop to exact square
            T.RandomHorizontalFlip(),                # augmentation
            T.ToTensor(),                            # PIL -> tensor [0,1]
            T.Normalize([0.5], [0.5]),               # scale to [-1, 1]
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = self.transform(sample["image"].convert("RGB"))
        caption = sample["text"]
        return image, caption


train_dataset = TextImageDataset(hf_dataset, resolution=RESOLUTION)
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH,
    shuffle=True,
    num_workers=2,
    drop_last=True,
)
print(f"DataLoader: {len(train_loader)} batches of {TRAIN_BATCH}")

In [ ]:
# ============================================================
# RELOAD THE PIPELINE IN FLOAT32 FOR TRAINING
# ============================================================
# Training needs float32 for stable gradients.
# We reload fresh to avoid any state from the float16 pipeline.

del pipe                                             # free the float16 pipeline
torch.cuda.empty_cache()                             # reclaim GPU memory

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipe = pipe.to(device)
pipe.safety_checker = None

vae = pipe.vae
unet = pipe.unet
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

print("Fresh pipeline loaded in float32 for training.")

In [ ]:
# ============================================================
# APPLY LoRA TO THE UNET
# ============================================================
# Same pattern as HuBERT and CLIP labs:
# 1. Define which layers to adapt (attention key/query/value projections)
# 2. Wrap the model with get_peft_model
# 3. Only the tiny LoRA matrices are trainable

lora_config = LoraConfig(
    r=LORA_R,                                        # rank of LoRA matrices
    lora_alpha=LORA_ALPHA,                           # scaling factor
    lora_dropout=LORA_DROPOUT,                       # dropout
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],  # attention layers
)

unet = get_peft_model(unet, lora_config)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"That is {total // trainable}x fewer parameters to train!")

# Freeze everything else
vae.requires_grad_(False)                            # VAE stays frozen
text_encoder.requires_grad_(False)                   # text encoder stays frozen

In [ ]:
# ============================================================
# SAVE A "BEFORE" IMAGE FOR COMPARISON
# ============================================================

test_prompt = "naruto character with spiky hair and orange outfit"

unet.eval()
pipe.unet = unet
before_image = pipe(
    test_prompt, num_inference_steps=30, guidance_scale=7.5,
    generator=torch.Generator(device).manual_seed(SEED),
).images[0]

plt.figure(figsize=(4, 4))
plt.imshow(before_image)
plt.title('BEFORE fine-tuning', fontsize=11)
plt.axis('off')
plt.show()

In [ ]:
# ============================================================
# TRAINING LOOP: same as Day 24 but in latent space
# ============================================================
# 1. Encode image with VAE → latent
# 2. Sample random timestep
# 3. Add noise to latent
# 4. Get text embedding from caption
# 5. UNet predicts noise (conditioned on text)
# 6. MSE loss between predicted and actual noise
#
# This is EXACTLY the same training you did in Day 24,
# just in latent space with text conditioning.

optimizer = optim.AdamW(unet.parameters(), lr=TRAIN_LR)  # only LoRA params have grad
losses = []
data_iter = iter(train_loader)                       # infinite data iterator

unet.train()
for step in tqdm(range(TRAIN_STEPS), desc="Training"):
    # Get next batch (loop back if exhausted)
    try:
        images, captions = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        images, captions = next(data_iter)

    images = images.to(device)                       # (B, 3, 256, 256) in [-1, 1]

    # Step 1: Encode images to latent space with the frozen VAE
    with torch.no_grad():
        latents = vae.encode(images).latent_dist.sample()  # (B, 4, 32, 32)
        latents = latents * vae.config.scaling_factor

    # Step 2: Sample random timesteps (one per image in batch)
    timesteps = torch.randint(0, scheduler.config.num_train_timesteps,
                              (TRAIN_BATCH,), device=device)

    # Step 3: Add noise to latents
    noise = torch.randn_like(latents)                # random noise
    noisy_latents = scheduler.add_noise(latents, noise, timesteps)  # x_t

    # Step 4: Encode captions to text embeddings
    tokens = tokenizer(list(captions), padding="max_length",
                       max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt")
    with torch.no_grad():
        text_emb = text_encoder(tokens.input_ids.to(device))[0]  # (B, 77, 768)

    # Step 5: UNet predicts the noise
    noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=text_emb).sample

    # Step 6: MSE loss (same as Day 24!)
    loss = F.mse_loss(noise_pred, noise)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)  # gradient clipping
    optimizer.step()

    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        avg = np.mean(losses[-500:])
        print(f"  Step {step+1}/{TRAIN_STEPS}  loss: {avg:.4f}")

plot_loss(losses, "LoRA Fine-Tuning Loss (MSE on noise prediction)")

In [ ]:
# ============================================================
# GENERATE WITH THE FINE-TUNED MODEL: Before vs After
# ============================================================

unet.eval()
pipe.unet = unet

test_prompts = [
    "naruto character with spiky hair and orange outfit",
    "a ninja with a red cape standing on a rooftop",
    "anime warrior with a glowing sword in a dark forest",
    "a young anime hero with blue eyes and a headband",
]

after_images = []
for i, p in enumerate(test_prompts):
    img = pipe(
        p, num_inference_steps=30, guidance_scale=7.5,
        generator=torch.Generator(device).manual_seed(SEED),
    ).images[0]
    after_images.append(img)

show_images(after_images, titles=test_prompts, ncols=4, figsize=(16, 5))

In [ ]:
# ============================================================
# SIDE BY SIDE: Before vs After fine-tuning
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(before_image)
axes[0].set_title('BEFORE LoRA', fontsize=12)
axes[0].axis('off')
axes[1].imshow(after_images[0])
axes[1].set_title('AFTER LoRA', fontsize=12)
axes[1].axis('off')
plt.suptitle(f'"{test_prompts[0]}"', fontsize=11)
plt.tight_layout()
plt.show()

---
## Wrap-up

Today we went from building diffusion from scratch to using and fine-tuning a real pretrained model. Here is how the pieces connect:

| | Day 24 (from scratch) | Today (pretrained SD) |
|---|---|---|
| **UNet** | Built Conv + skip connections (~12M params) | Used pretrained UNet (~860M params) |
| **VAE** | Built encoder/decoder | Used pretrained latent-space VAE |
| **Conditioning** | None (unconditional) | Text via CLIP embeddings |
| **Fine-tuning** | Trained from random weights | LoRA on attention layers (<1% params) |
| **Resolution** | 64×64 | 256×256 |
| **Training** | MSE on predicted noise | Exactly the same — MSE on predicted noise |

The core idea never changed: **train a network to predict noise, then iteratively remove it**. Everything else — VAE compression, text conditioning, LoRA — is engineering on top of that one idea.

**Next up: the competition.** You now have all the tools. Can you make Stable Diffusion swap what it thinks a zebra and a giraffe look like?